In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
%config Completer.use_jedi = True

In [4]:
from langchain_openai import OpenAI

In [5]:
os.environ["OPENAI_API_KEY"]= os.getenv("chatgpt_key")

In [8]:
llm = OpenAI(temperature=0.6)
name = llm.invoke("I want to open an Italian restaurant. Please suggest a fency name for this.")
print(name)



"La Dolce Vita Trattoria" 


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.7)

prompt = PromptTemplate.from_template(
    "I want to open a {cuisine} restaurant. Suggest 5 fancy names."
)

chain = prompt | llm

response = chain.invoke({"cuisine": "Mexican"})

print(response.content)

1. Tequila Sunrise
2. Azul Agave
3. La Cucaracha
4. El Sol y la Luna
5. Pico de Oro


In [14]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

parser = JsonOutputParser()

prompt = PromptTemplate.from_template(
    """
    Suggest a restaurant idea for {cuisine}.
    Return in JSON format:
    {{
        "name": "",
        "menu": [],
        "location": ""
    }}
    """
)

chain = prompt | llm | parser

result = chain.invoke({"cuisine": "Italian"})

print(result)

{'name': 'Rustic Trattoria', 'menu': ['Homemade pasta dishes', 'Wood-fired pizzas', 'Antipasti platters', 'Tiramisu dessert'], 'location': 'Downtown area with outdoor patio seating'}


In [23]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
    give me top 2 rated Restaurants in the Toronto Downtown for the {cuisine} with it locations
    Return strictly in JSON format:
{{
    "restaurants": [
        {{
            "name": "",
            "location": ""
        }}
    ]
}}
    """)

chain = prompt | llm | parser

restaurant_names = chain.invoke({"cuisine":"Pakistani"})





In [26]:
restaurant_lists = [r['name'] for r in restaurant_names['restaurants']]
print(restaurant_lists)

[' Lahore Tikka House', ' Karachi Kitchen']


In [29]:
menu_prompt = PromptTemplate.from_template(
"""
For the restaurant "{restaurant_name}", give top 3 popular menu items.

Return in JSON:
{{
  "restaurant": "{restaurant_name}",
  "menu_items": ["", "", ""]
}}
"""
)

menu_chain = menu_prompt | llm | parser

output = [];

for restuarant in restaurant_lists:
    output.append(menu_chain.invoke({"restaurant_name": restuarant}))

print(output)

    

[{'restaurant': 'Lahore Tikka House', 'menu_items': ['Chicken Tikka', 'Seekh Kebab', 'Biryani']}, {'restaurant': 'Karachi Kitchen', 'menu_items': ['Chicken Biryani', 'Nihari', 'Seekh Kabab']}]


In [41]:
restaurant_lists =  [x.strip() for x in restaurant_lists]
print(restaurant_lists)

['Lahore Tikka House', 'Karachi Kitchen']


In [48]:
menu_prompt = PromptTemplate.from_template(
"""
You are given a list of restaurants.

{restaurants}

provide top 10 rated menus for each restaurants with ratings

Return in JSON:
{{
  "results": [ 
  "restaurant": "",
  "menu_items": ["", "", ""]
  ]
}}
"""
)

menu_chain = menu_prompt | llm | parser

output = [];

output = menu_chain.invoke({"restaurants": restaurant_lists})

print(output)

    

{'results': [{'restaurant': 'Lahore Tikka House', 'menu_items': [{'menu_item': 'Chicken Biryani', 'rating': 4.5}, {'menu_item': 'Butter Chicken', 'rating': 4.3}, {'menu_item': 'Seekh Kebab', 'rating': 4.2}, {'menu_item': 'Garlic Naan', 'rating': 4.0}, {'menu_item': 'Palak Paneer', 'rating': 4.0}, {'menu_item': 'Samosa', 'rating': 3.8}, {'menu_item': 'Masala Dosa', 'rating': 3.7}, {'menu_item': 'Dal Makhani', 'rating': 3.5}, {'menu_item': 'Gulab Jamun', 'rating': 3.4}, {'menu_item': 'Aloo Paratha', 'rating': 3.3}]}, {'restaurant': 'Karachi Kitchen', 'menu_items': [{'menu_item': 'Chicken Biryani', 'rating': 4.6}, {'menu_item': 'Nihari', 'rating': 4.4}, {'menu_item': 'Haleem', 'rating': 4.3}, {'menu_item': 'Chapli Kebab', 'rating': 4.2}, {'menu_item': 'Bhindi Masala', 'rating': 4.1}, {'menu_item': 'Palak Gosht', 'rating': 4.0}, {'menu_item': 'Sheer Khurma', 'rating': 3.9}, {'menu_item': 'Aloo Puri', 'rating': 3.8}, {'menu_item': 'Shami Kebab', 'rating': 3.7}, {'menu_item': 'Gajar Halwa', 